# STT_TTS Task 


### Testing voice recording to a File (**DONE**)

**Requirements**
-   Download sounddevice library    [✔️]

**Notice** <br />
Uncomment the following lines to list all the device you have: 
- _print (sd.query_devices())_<br />
- _print (sd.default.device)_<br />
<br />

Uncomment the following lines to choose (Input, Output) Devices' Index:<br />
- _sd.default.device = ([  ], [  ])_  **They should have the same sampling rate** <br />
<br />



In [ ]:
#Finding ID of Audio Devices
import sounddevice as sd 
import numpy as np
import wave
# print (sd.query_devices())
# print (sd.default.device)

# Selecting an Audio Device by ID
sd.default.device = (8, 10)    # (input device ID, output device ID)

# Finding supported Sample Rates of Audio Devices
device_info_input = sd.query_devices(8) 
print ("Input Sample Rate is: ",device_info_input['default_samplerate']) # Sample Rate of Input Device

device_info_output = sd.query_devices(10) 
print ("Output Sample Rate is: ",device_info_output['default_samplerate']) # Sample Rate of Output Device

# Testing if the mic is working 
duration = 6  # seconds
print("Recording...")

# Record audio (settings: duration, sample rate, channels)
myrecording = sd.rec(int(duration * device_info_input['default_samplerate']), samplerate=device_info_input['default_samplerate'], channels=1)
sd.wait()  # Wait
print("Recording complete \n Max recorded Amp = ", np.max(myrecording)) # Print the maximum amplitude between -1.0 and 1.0

# Save the recording to a WAV file
with wave.open('recorded_input.wav', 'wb') as wf:
    wf.setnchannels(1)  # Mono
    wf.setsampwidth(2)  # 16 bits per sample
    wf.setframerate(int(device_info_input['default_samplerate']))
    wf.writeframes((myrecording * 32767).astype(np.int16).tobytes())

print("Audio saved to recorded_input.wav")


### Testing Keyboard Record Control (**DONE**)

**Requirements**
-   Downoad  numpy          [✔️]
-   Download keyboard       [✔️]

In [ ]:
import sounddevice as sd
import numpy as np
import wave
import keyboard  # pip install keyboard
import time

# Device setup
sd.default.device = (8, 10)  # (input device ID, output device ID)
device_info_input = sd.query_devices(8) 
sample_rate = int(device_info_input['default_samplerate'])

print("Hold SPACE to record...")

# frames parameter is a list to store recorded audio chunks
frames = []
blocksize = 1024

# Callback function to capture audio in chunks
def callback(indata, frames_count, time_info, status):
    frames.append(indata.copy())

# Wait for spacebar press
keyboard.wait('space')
print("Recording...")
stream = sd.InputStream(samplerate=sample_rate, channels=1, callback=callback, blocksize=blocksize) # Start audio stream 
stream.start() # Start recording

# Keep recording while spacebar is pressed
while keyboard.is_pressed('space'):
    time.sleep(0.01)

stream.stop()
stream.close()
print("Recording complete.")

# Combine frames
recording = np.concatenate(frames, axis=0)

# Save to WAV
with wave.open('recorded_input.wav', 'wb') as wf:
    wf.setnchannels(1)
    wf.setsampwidth(2)
    wf.setframerate(sample_rate)
    wf.writeframes((recording * 32767).astype(np.int16).tobytes())

print("Audio saved to recorded_input.wav")


### Testing Automatic Stops (**Done**)

In [ ]:
import sounddevice as sd
import numpy as np
import wave
import keyboard
import time

# Device setup
sd.default.device = (8, 10)  # (input, output)
device_info_input = sd.query_devices(8)
sample_rate = int(device_info_input['default_samplerate'])

# Parameters
blocksize = 1024
silence_threshold = 0.01   # adjust if needed
silence_duration = 2.0     # seconds

frames = []
silence_time = 0.0

print("Press SPACE to start recording...")
keyboard.wait('space')
print("Recording...")

def rms(data):
    return np.sqrt(np.mean(np.square(data)))

def callback(indata, frames_count, time_info, status):
    global silence_time
    frames.append(indata.copy())
    if rms(indata) < silence_threshold:
        silence_time += blocksize / sample_rate
    else:
        silence_time = 0.0

stream = sd.InputStream(samplerate=sample_rate, channels=1, callback=callback, blocksize=blocksize)
stream.start()

while silence_time < silence_duration:
    if not keyboard.is_pressed('space') and len(frames) == 0:
        # user never started speaking
        time.sleep(0.01)
        continue
    time.sleep(0.05)

stream.stop()
stream.close()
print("Silence detected — recording stopped.")

# Combine frames
recording = np.concatenate(frames, axis=0)

# Save to WAV
with wave.open('recorded_input.wav', 'wb') as wf:
    wf.setnchannels(1)
    wf.setsampwidth(2)
    wf.setframerate(sample_rate)
    wf.writeframes((recording * 32767).astype(np.int16).tobytes())

print("Audio saved to recorded_input.wav")


### Testing Automatic Start-Stop (**DONE**)

In [1]:
import sounddevice as sd
import numpy as np
import wave
import time

# Device setup
sd.default.device = (8, 10)  # (input, output)
device_info_input = sd.query_devices(8)
sample_rate = int(device_info_input['default_samplerate'])

# Parameters
blocksize = 1024
silence_threshold = 0.01   # noise floor
silence_duration = 2.0     # seconds of silence to stop
sound_trigger = 0.02       # sound level to start
frames = []
recording_started = False
silence_time = 0.0

def rms(data):
    return np.sqrt(np.mean(np.square(data)))


# Callback function to process audio chunks
def callback(indata, frames_count, time_info, status):
    global recording_started, silence_time
    level = rms(indata)

    if recording_started:
        frames.append(indata.copy())
        if level < silence_threshold:
            silence_time += blocksize / sample_rate
        else:
            silence_time = 0.0
    else:
        if level > sound_trigger:
            recording_started = True
            frames.append(indata.copy())
            silence_time = 0.0
            print("Sound detected — recording started.")

print("Waiting for sound...")
stream = sd.InputStream(samplerate=sample_rate, channels=1, callback=callback, blocksize=blocksize)
stream.start()

while not recording_started:
    time.sleep(0.05)

while silence_time < silence_duration:
    time.sleep(0.05)

stream.stop()
stream.close()
print("Silence detected — recording stopped.")

recording = np.concatenate(frames, axis=0)

with wave.open('recorded_input.wav', 'wb') as wf:
    wf.setnchannels(1)
    wf.setsampwidth(2)
    wf.setframerate(sample_rate)
    wf.writeframes((recording * 32767).astype(np.int16).tobytes())

print("Audio saved to recorded_input.wav")


Waiting for sound...
Sound detected — recording started.
Silence detected — recording stopped.
Audio saved to recorded_input.wav
